# 8.3 · Compact sets: bit vectors and Bloom filters

*Systems and scale* · 17 exercises

Represent a set with one bit per possible member, or approximately in far less with a Bloom filter.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-8.3) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-8.3) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-8.3)

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import hashlib
import math
import random
import sys

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    class _Raises:
        def __init__(self, kind):
            self.kind = kind

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            return issubclass(kind, self.kind)

    class pytest:  # noqa: N801
        raises = _Raises

### 8.3.1 · A set as a boolean array  `Core`

**Task.** Write `BoolArraySet(universe)` for integers 0..universe-1 backed by a list of booleans, with `add`, `remove`, `__contains__` and `__len__`.

**Example.** universe 10: add 3, add 7, remove 3 -> `7 in s` is `True`, `3 in s` is `False`, `len(s)` is `1`

**Big O.** O(1) per operation, but one Python object slot (8 bytes) per possible value.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-1)

In [ ]:
# 8.3.1 A set as a boolean array
class BoolArraySet:
    def __init__(self, universe):
        raise NotImplementedError

    def add(self, x):
        raise NotImplementedError

    def remove(self, x):
        raise NotImplementedError

    def __contains__(self, x):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

### 8.3.2 · A bit vector with masks  `Core`

**Task.** Write `BitVector(size)` on a `bytearray` of ceil(size / 8) bytes. Value x lives in byte `x >> 3`, bit `x & 7`. Implement `add` (OR with the mask), `remove` (AND with the inverted mask) and `__contains__` (AND, then test).

**Example.** `bv = BitVector(100); bv.add(42)` -> `42 in bv` is `True`; `bv.data[5]` is `4` (bit 2 of byte 5)

**Big O.** O(1) per operation, and 1 bit per possible value: 64 times less than a list of booleans.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-2)

In [ ]:
# 8.3.2 A bit vector with masks
class BitVector:
    def __init__(self, size):
        raise NotImplementedError

    def add(self, x):
        raise NotImplementedError

    def remove(self, x):
        raise NotImplementedError

    def __contains__(self, x):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
def vector(size, members):
    bv = BitVector(size)
    for x in members:
        bv.add(x)
    return bv

In [ ]:
def test_ex_8_3_1():
    s = BoolArraySet(10)
    s.add(3)
    s.add(7)
    s.remove(3)
    s.add(7)
    assert 7 in s and 3 not in s and len(s) == 1


test_ex_8_3_1()
print('8.3.1 passed')

In [ ]:
def test_ex_8_3_2():
    bv = BitVector(100)
    bv.add(42)
    assert 42 in bv and 41 not in bv and bv.data[5] == 4
    bv.remove(42)
    assert 42 not in bv
    bv.remove(42)
    assert 42 not in bv


test_ex_8_3_2()
print('8.3.2 passed')

### 8.3.3 · Count the members  `Core`

**Task.** Add `__len__` that counts set bits across all bytes, using `int.bit_count` on each byte (or a 256-entry lookup table).

**Example.** add 1, 5, 99 -> `len(bv)` is `3`

**Big O.** O(size / 8). Keep a running counter instead if you need O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-3)

In [ ]:
# 8.3.3 Count the members
def __len__(self):
    raise NotImplementedError


BitVector.__len__ = __len__

In [ ]:
def test_ex_8_3_3():
    bv = vector(100, (1, 5, 99))
    assert len(bv) == 3
    bv.add(5)
    assert len(bv) == 3 and len(BitVector(10)) == 0


test_ex_8_3_3()
print('8.3.3 passed')

### 8.3.4 · Union, intersection and difference  `Core`

**Task.** Add `union`, `intersection` and `difference`, each returning a new `BitVector` of the same size by combining the bytes with `|`, `&` and `& ~`.

**Example.** {1, 2, 3} and {2, 3, 4} -> union `{1, 2, 3, 4}`, intersection `{2, 3}`, difference `{1}`

**Big O.** O(size / 8): eight set members combined per byte operation.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-4)

In [ ]:
# 8.3.4 Union, intersection and difference
def _combine(self, other, op):
    raise NotImplementedError

def union(self, other):
    raise NotImplementedError

def intersection(self, other):
    raise NotImplementedError

def difference(self, other):
    raise NotImplementedError


BitVector._combine = _combine
BitVector.union = union
BitVector.intersection = intersection
BitVector.difference = difference

In [ ]:
def test_ex_8_3_4():
    a, b = vector(10, (1, 2, 3)), vector(10, (2, 3, 4))
    u, i, d = a.union(b), a.intersection(b), a.difference(b)
    assert [x for x in range(10) if x in u] == [1, 2, 3, 4]
    assert [x for x in range(10) if x in i] == [2, 3]
    assert [x for x in range(10) if x in d] == [1]


test_ex_8_3_4()
print('8.3.4 passed')

### 8.3.5 · Iterate over members  `Core`

**Task.** Add `__iter__` yielding members in increasing order, skipping zero bytes entirely.

**Example.** add 9, 2, 63 -> `list(bv)` is `[2, 9, 63]`

**Big O.** O(size / 8 + members).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-5)

In [ ]:
# 8.3.5 Iterate over members
def __iter__(self):
    raise NotImplementedError


BitVector.__iter__ = __iter__

In [ ]:
def test_ex_8_3_5():
    assert list(vector(70, (9, 2, 63))) == [2, 9, 63]
    members = set(random.Random(3).sample(range(5000), 700))
    big = vector(5000, members)
    assert list(big) == sorted(members) and len(big) == 700


test_ex_8_3_5()
print('8.3.5 passed')

### 8.3.6 · Estimate the space  `Core`

**Task.** Write `space_report(n)` returning the bytes used by a Python `set` of 0..n-1 (the set plus its int objects), a list of n booleans, and a `BitVector(n)`.

**Example.** n = 1,000,000 -> set about 60 MB, list about 8 MB, bit vector `125000` bytes

**Big O.** O(N) each, but the constants differ by a factor of about 500.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-6)

In [ ]:
# 8.3.6 Estimate the space
def space_report(n):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_6():
    rep = space_report(100_000)
    assert rep["bit_vector"] == 12500 and rep["bit_vector"] * 50 < rep["bool_list"] < rep["set"]


test_ex_8_3_6()
print('8.3.6 passed')

### 8.3.7 · Find duplicates in a bounded range  `Core`

**Task.** Write `first_duplicate(nums, universe)` returning the first value seen twice, using a `BitVector` as the "seen" set.

**Example.** `first_duplicate([4, 9, 1, 9, 4], 10)` -> `9`; no repeats -> `None`

**Big O.** O(N) time, universe / 8 bytes of space, no matter how many numbers there are.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-7)

In [ ]:
# 8.3.7 Find duplicates in a bounded range
def first_duplicate(nums, universe):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_7():
    assert first_duplicate([4, 9, 1, 9, 4], 10) == 9 and first_duplicate([1, 2, 3], 10) is None


test_ex_8_3_7()
print('8.3.7 passed')

### 8.3.8 · A built-in type as a growable bit set  `Practice`

**Task.** Write `IntBitSet` holding one Python int: `add` with `|=`, `__contains__` with `>>` and `&`, `__or__` and `__and__` for set operations, and `__len__` with `bit_count`.

**Example.** `a = IntBitSet([1, 5]); b = IntBitSet([5, 7])` -> `list(a | b)` is `[1, 5, 7]`; `len(a & b)` is `1`

**Big O.** Big-int operations run in O(size / 64) machine words, and the set grows as needed.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-8)

In [ ]:
# 8.3.8 A built-in type as a growable bit set
class IntBitSet:
    def __init__(self, items=()):
        raise NotImplementedError

    def add(self, x):
        raise NotImplementedError

    def __contains__(self, x):
        raise NotImplementedError

    def __or__(self, other):
        raise NotImplementedError

    def __and__(self, other):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __iter__(self):
        raise NotImplementedError

In [ ]:
def test_ex_8_3_8():
    x, y = IntBitSet([1, 5]), IntBitSet([5, 7])
    assert list(x | y) == [1, 5, 7] and len(x & y) == 1 and 5 in x and 7 not in x


test_ex_8_3_8()
print('8.3.8 passed')

### 8.3.9 · Missing numbers from a range  `Practice`

**Task.** Write `missing(nums, n)` returning every value in 0..n-1 that does not appear in `nums`, marking a `BitVector` and then scanning it.

**Example.** `missing([0, 3, 1, 6], 7)` -> `[2, 4, 5]`

**Big O.** O(N + n) time, n / 8 bytes of space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-9)

In [ ]:
# 8.3.9 Missing numbers from a range
def missing(nums, n):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_9():
    assert missing([0, 3, 1, 6], 7) == [2, 4, 5] and missing(list(range(5)), 5) == []


test_ex_8_3_9()
print('8.3.9 passed')

In [ ]:
# Support code used by the exercises: run it as is.
def _two_hashes(item):
    d = hashlib.sha256(str(item).encode()).digest()
    return int.from_bytes(d[:8], "big"), int.from_bytes(d[8:16], "big") | 1

### 8.3.10 · k hash functions from two  `Core`

**Task.** Write `bit_positions(item, k, m)` returning k bit positions in [0, m) using double hashing: `(h1 + i * h2) % m` for i in 0..k-1, where h1 and h2 come from one SHA-256 digest.

**Example.** `bit_positions("cat", 3, 32)` -> three positions in 0..31, the same on every run

**Big O.** O(k) after one hash computation. Kirsch and Mitzenmacher showed this behaves as well as k independent hashes.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-10)

In [ ]:
# 8.3.10 k hash functions from two
def bit_positions(item, k, m):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_10():
    assert bit_positions("cat", 3, 32) == bit_positions("cat", 3, 32)
    assert len(bit_positions("cat", 3, 32)) == 3 and all(0 <= p < 32 for p in bit_positions("cat", 3, 32))


test_ex_8_3_10()
print('8.3.10 passed')

### 8.3.11 · A Bloom filter  `Core`

**Task.** Write `BloomFilter(m, k)` with an m-bit `bytearray`, `add(item)` setting the item's k bits, and `might_contain(item)` returning True only if all k bits are set.

**Example.** add "apple" and "pear" -> `might_contain("apple")` is `True`; `might_contain("grape")` is almost certainly `False`

**Big O.** O(k) per operation; m bits of memory regardless of how large the items are.

**Note.** "No" is always right. "Yes" means "probably": other items may have set those same bits.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-11)

In [ ]:
# 8.3.11 A Bloom filter
class BloomFilter:
    def __init__(self, m, k):
        raise NotImplementedError

    def _set(self, i):
        raise NotImplementedError

    def _get(self, i):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def might_contain(self, item):
        raise NotImplementedError

In [ ]:
def test_ex_8_3_11():
    bf = BloomFilter(1000, 5)
    bf.add("apple")
    bf.add("pear")
    assert bf.might_contain("apple") and bf.might_contain("pear") and not bf.might_contain("grape")
    bf2 = BloomFilter(5000, 4)
    for i in range(300):
        bf2.add(f"w{i}")
    assert all(bf2.might_contain(f"w{i}") for i in range(300))


test_ex_8_3_11()
print('8.3.11 passed')

### 8.3.16 · Union of two filters  `Practice`

**Task.** Add `union(other)` returning a new filter whose bits are the OR of both, for filters built with the same m and k.

**Example.** filter A holds "x", filter B holds "y" -> the union might contain both

**Big O.** O(m / 8). Handy for merging filters built on separate machines.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-16)

In [ ]:
# 8.3.16 Union of two filters
def union(self, other):
    raise NotImplementedError


BloomFilter.union = union

In [ ]:
def test_ex_8_3_16():
    a, b = BloomFilter(512, 3), BloomFilter(512, 3)
    a.add("x")
    b.add("y")
    u = a.union(b)
    assert u.might_contain("x") and u.might_contain("y")


test_ex_8_3_16()
print('8.3.16 passed')

### 8.3.12 · Predict the false-positive rate  `Core`

**Task.** Write `false_positive_rate(n, m, k)` = `(1 - e^(-k n / m))^k`, the chance a never-added item looks present after n insertions.

**Example.** `false_positive_rate(1000, 10_000, 7)` -> about `0.0082` (under 1%)

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-12)

In [ ]:
# 8.3.12 Predict the false-positive rate
def false_positive_rate(n, m, k):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_12():
    assert abs(false_positive_rate(1000, 10_000, 7) - 0.0082) < 0.0005


test_ex_8_3_12()
print('8.3.12 passed')

### 8.3.13 · Size a filter for a target error  `Core`

**Task.** Write `optimal_params(n, p)` returning `(m, k)`: bits `m = ceil(-n ln p / (ln 2)^2)` and hash count `k = round((m / n) ln 2)`.

**Example.** `optimal_params(1_000_000, 0.01)` -> `(9585059, 7)`: about 1.2 MB for a million items at 1% error

**Big O.** O(1). Each tenfold cut in p costs only about 4.8 more bits per item.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-13)

In [ ]:
# 8.3.13 Size a filter for a target error
def optimal_params(n, p):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_13():
    assert optimal_params(1_000_000, 0.01) == (9585059, 7)


test_ex_8_3_13()
print('8.3.13 passed')

### 8.3.14 · Measure the real error  `Core`

**Task.** Write `measured_fp_rate(n, m, k, probes)`: add the strings "in-0" .. "in-{n-1}", then probe "out-0" .. "out-{probes-1}" (never added) and return the fraction reported present.

**Example.** `measured_fp_rate(1000, 10_000, 7, 20_000)` lands close to the 1.3.3 prediction (within about 0.005)

**Big O.** O((n + probes) * k).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-14)

In [ ]:
# 8.3.14 Measure the real error
def measured_fp_rate(n, m, k, probes):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_14():
    measured = measured_fp_rate(1000, 10_000, 7, 20_000)
    assert abs(measured - 0.0082) < 0.005


test_ex_8_3_14()
print('8.3.14 passed')

### 8.3.15 · Duplicate detection in a stream  `Core`

**Task.** Write `stream_duplicates(items, m, k)` that reports items the filter says it has seen before. Then write `confirmed_duplicates(items, m, k)` that double-checks each report against an exact set, and count how many reports were false.

**Example.** a stream of 5,000 unique items plus 100 repeats with m = 64,000, k = 6 -> all 100 repeats reported, and only a handful of false alarms

**Big O.** O(N * k) time, m bits of space. Real systems (web crawlers, databases) use the filter to skip most exact lookups.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-15)

In [ ]:
# 8.3.15 Duplicate detection in a stream
def stream_duplicates(items, m, k):
    raise NotImplementedError


def confirmed_duplicates(items, m, k):
    raise NotImplementedError

In [ ]:
def test_ex_8_3_15():
    items = [f"u{i}" for i in range(5000)] + [f"u{i}" for i in range(0, 5000, 50)]
    reported = stream_duplicates(items, 64_000, 6)
    real, false_alarms = confirmed_duplicates(items, 64_000, 6)
    assert len(real) == 100 and false_alarms < 20 and len(reported) >= 100


test_ex_8_3_15()
print('8.3.15 passed')

### 8.3.17 · A counting Bloom filter that can delete  `Stretch`

**Task.** Write `CountingBloomFilter(m, k)` with a list of small counters instead of bits: `add` increments, `remove` decrements (only for items you added), and `might_contain` checks all counters are above zero.

**Example.** add "a", add "b", remove "a" -> `might_contain("a")` is `False` (barring collisions), `might_contain("b")` is `True`

**Big O.** O(k) per operation, at the cost of 4 to 8 bits per counter instead of 1.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-8-3-17) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-8-3-17)

In [ ]:
# 8.3.17 A counting Bloom filter that can delete
class CountingBloomFilter:
    def __init__(self, m, k):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def remove(self, item):
        raise NotImplementedError

    def might_contain(self, item):
        raise NotImplementedError

In [ ]:
def test_ex_8_3_17():
    cb = CountingBloomFilter(1000, 4)
    cb.add("a")
    cb.add("b")
    assert cb.remove("a") and not cb.might_contain("a") and cb.might_contain("b") and not cb.remove("zzz")


test_ex_8_3_17()
print('8.3.17 passed')